## 实战: 多功能智能助手

项目需求: 开发一个多功能智能助手, 支持:
1. 天气查询: 查询城市天气
2. 数学计算: 复杂数学运算
3. 时间查询: 获取当前时间, 日期计算
4. 货币转换: 多种货币之间转换
5. 信息搜索: 搜索产品, 新闻等信息

In [1]:
# 模型初始化
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 获取环境变量
api_key = os.getenv("ZHIPUAI_API_KEY")
base_url = os.getenv("ZHIPUAI_API_BASE_URL")

# 模型初始化
model = init_chat_model(
    model="glm-5.3",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)

In [ ]:
# 创建智能体, 调用工具
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import SystemMessage
from langchain.tools import tool
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from rich import print as rprint
from typing import Union

# tools
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """ 
    查询某个城市的天气

    Args:
        city (str): 城市名称

    Returns: 
        str: 返回城市天气
    """

    return f"{city}天气晴朗"

@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串, 例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串, 包含客户姓名, 等级, 最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录: 张三, VIP 客户, 最近购买日期: 2026-01-15, 累计消费: $15,000"
    elif "李四" in query.lower():
        return "客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200"
    else:
        return f"关于客户{query}, 无记录"
    
# 内置工具
tavily_search = TavilySearch(
    max_results=1,
    tavily_api_key=os.getenv("TAVILY_API_KEY")
)

# 使用 ToolStrategy 结构化输出
class WeatherInfo(BaseModel):
    """天气信息"""
    city: str = Field(description="城市")
    temperature: float = Field(description="温度, 单位摄氏度")
    condition: str = Field(description="天气情况")

class TextReply(BaseModel):
    reason: str = Field(description="为什么不能返回结构化数据")
    text: str = Field(description="自然语言回复")


agent = create_agent(
    model=model,
    tools=[get_weather, search_customer_database, tavily_search],
    system_prompt=SystemMessage("""你是一个多功能智能助手。请仔细分析用户的每一个请求。

# 结构化输出规则
1. **只有**在用户的请求是**单一**的“查询某个城市天气”时，你才能返回 `WeatherInfo` 结构。
2. 如果用户的请求包含**多个**任务（例如，同时问天气和新闻），或者**不是**查询天气的任务，你必须返回 `TextReply` 结构。
3. 在 `TextReply` 的 `reason` 字段中，简要说明你为何不返回天气信息（例如："用户请求包含多个任务"）。
4. 在 `TextReply` 的 `text` 字段中，用自然语言逐一、完整地回答用户的所有问题。"""),
    response_format=ToolStrategy(
        Union[WeatherInfo, TextReply],
        handle_errors=True,
        # 将最后一条 ToolMessage 的 content 改为简单文字输出, 默认情况下返回 Returning structured response: + 响应的所有内容
        tool_message_content="数据提取成功"
    )
)

# response = agent.invoke({
#     "messages": [{
#         "role": "user",
#         "content": f"杭州今天晴天, 15摄氏度, 小明出门问小朋友, 今天有什么新闻, 黄金今天多少钱一克, 换算汇率, 美元和人民币"
#     }]
# })
# rprint(response)

response_stream = agent.stream({
    "messages": [{
        "role": "user",
        "content": f"杭州今天晴天, 15摄氏度, 小明出门问小朋友, 今天有什么新闻, 黄金今天多少钱一克, 换算汇率, 美元和人民币"
    }]
})

for chunk in response_stream:
    rprint(chunk)
    print("-"*50)


{
    'model': {
        'messages': [
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 1034,
                        'prompt_tokens': 2100,
                        'total_tokens': 3134,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 914,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 2048}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '2026111714111391a2ec1997d14406',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1e821-84d0-7423-b0c3-8244a55d5ef5-0',
                tool_calls=[
                    {
                        'name': 'tavily_search',
                        'args': {
                            'query': '今日重要新闻 头条',
                            'search_depth': 'fast',
                            'time_range': 'day',
                            'topic': 'news'
                        },
                        'id': 'call_-7079982454760535079',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'tavily_search',
                        'args': {
                            'query': '今日黄金价格 多少钱一克',
                            'search_depth': 'fast',
                            'time_range': 'day',
                            'topic': 'finance'
                        },
                        'id': 'call_-7079982454760535078',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'tavily_search',
                        'args': {
                            'query': '美元兑人民币汇率 今日',
                            'search_depth': 'fast',
                            'time_range': 'day',
                            'topic': 'finance'
                        },
                        'id': 'call_-7079982454760535077',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 2100,
                    'output_tokens': 1034,
                    'total_tokens': 3134,
                    'input_token_details': {'cache_read': 2048},
                    'output_token_details': {'reasoning': 914}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content="{'error': ValueError('Error 400: Finance topic is not supported for fast or ultra-fast 
search_depth.')}",
                name='tavily_search',
                id='ca548ef4-5e90-427a-b6e7-55053b9e6f0e',
                tool_call_id='call_-7079982454760535078'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content="{'error': ValueError('Error 400: Finance topic is not supported for fast or ultra-fast 
search_depth.')}",
                name='tavily_search',
                id='16cbcb97-1273-4379-bb28-7b2305e1243e',
                tool_call_id='call_-7079982454760535077'
            )
        ]
    }
}

--------------------------------------------------


... (6 outputs left out)

{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"query": "美元兑人民币汇率 今日", "follow_up_questions": null, "answer": null, "images": 
[], "results": [{"url": 
"https://hk.finance.yahoo.com/news/%E6%91%A9%E6%A0%B9%E8%B3%87%E7%AE%A1%E6%96%99%E4%BA%BA%E6%B0%91%E5%B9%A3%E6%9C%8
9%E6%9C%9B%E6%96%BC%E6%9C%AA%E4%BE%8610-15%E5%B9%B4%E5%8D%87%E5%80%BC%E8%87%B35-3%E6%B0%B4%E5%B9%B3-081554602.html"
, "title": "摩根資管料人民幣有望於未來10-15年升值至5.3水平", "content": "# 
摩根資管料人民幣有望於未來10-15年升值至5.3水平. <匯港通訊> 
摩根資產管理亞洲多元資產方案首席投資組合經理盛楠於記者會認為，人民銀行短期內不希望人民幣升值太快，惟因為美國通脹料
持續高企，美元有貶值壓力，反之中國通脹較低，而政策制定者亦希望推動人民幣國際化，加上中國人均收入提升，所以預期，未
來10-15年人民幣匯價會繼續升值，美元兌人民幣料會由約6.7變至5.3。. 
另外，摩根大通私人銀行亞洲投資及協作部主管劉君博表示，其客戶均希望低息貸款，而港元與人民幣利率較低，加上匯率波動不
大，所以成為借貸的貨幣。. 
摩根資產管理今日發佈《2027年長期資本市場假設》，為投資者提供各種資產類別未來10-15年風險回報展望。通脹不確定性及市場
波幅仍然偏高，此報告旨在為投資者構建穩健的投資組合提供切實可行的見解。. 
本年度報告為第31期，預測未來10-15年美元60/40股債投資組合年回報率將升至7.0%，但股債相關性上升，意味著投資者需要進一
步分散投資。縱使勞動力及人口結構限制為長期經齊增長帶來壓力，報告指出，各行業更廣泛地投資及採用 
Al，將在短期內提振企業盈利，長遠有望提升生產力。報告亦指出，投資者可透過配置不同地區股票及另類投資，尤其是對沖基金
及實物資產，提升投資組合的多元化程度。對於採用「60/40+」投資組合一—將30%資產配置於多元化另類投資的投資者，預測回報
率將升至7.5%，夏普比率亦較傳統60/40策略高出21%。. 報告又指出，美國企業利潤率超過15%，帶動動美國大型股回報預測上調80
個基點至7.5%，發達市場股票預測回報率亦有史以來首次超越新興市場。隨著實質孳息率上升，環球綜合債券回報預測由2021年低
位2.1%升至目前5.3%，創20年來新高。. 
摩根資產管理亞太區首席市場策略師許長泰表示，該行短期內仍偏好美、歐、日等成熟市場，惟料再過兩年，新興市場出現回吐，
估值變得便宜，或會吸引資流向新興市場。. 
此外，報告將美國大型股的預期回報率由6.7%(美元)上調至7.5%(美元)，反映科技應用持續普及及生產力提升。不過，起始估值偏
高及投資集中度較高，仍是需要留意的因素。美國具備有利條件，帶領 AI 應用發展。 （BC).", "score": 0.5001094, 
"raw_content": null, "id": "6663ff-00"}], "response_time": 0.0, "request_id": 
"5ea00e0b-ce8b-4489-a783-70277975793f"}',
                name='tavily_search',
                id='65232427-a449-4836-9632-93a187dbadb2',
                tool_call_id='call_-7079981527047595588'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 978,
                        'prompt_tokens': 4603,
                        'total_tokens': 5581,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 607,
                            'rejected_prediction_tokens': None
                        },
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 2624}
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5.3',
                    'system_fingerprint': None,
                    'id': '202611171411440af8f6cdb75f4ef7',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a1e821-fcc1-7703-8f8f-b12148f0c904-0',
                tool_calls=[
                    {
                        'name': 'TextReply',
                        'args': {
                            'reason': 
'用户的请求包含多个任务（天气、新闻、黄金价格、汇率换算），并非单一的天气查询，因此不返回 WeatherInfo 
结构化数据。',
                            'text': 
'好的，我来逐一回答您的所有问题：\n\n【今日天气】\n杭州今天天气晴朗，气温15摄氏度，非常适合外出活动。小明可以放心出
门和小朋友们一起玩。\n\n【今日新闻】\n根据最新搜索到的新闻：\n1. 
美国方面：美国国会正在审议一项审批法案，该法案将加快对输电线路、天然气管道等多种能源项目的联邦环境审查，同时要求数
据中心为消耗的电力支付数十亿美元的费用，以便让开发商更容易获得运行数据中心所需的电力。\n\n【今日黄金价格】\n国际金
价目前徘徊在14个月低位附近：\n- 现货黄金约每盎司3,845美元（下跌0.3%）\n- 换算下来约为每克123.6美元\n- 
按汇率折算约合人民币每克约828元（国际现货价）\n- 
国内品牌金饰零售价通常会在此基础上加上加工费和品牌溢价，实际购买价格会更高一些\n金价走低的主要原因是美国通胀担忧挥
之不去、市场对美联储进一步收紧货币政策的预期升温，加之科技股下跌带来的压力。\n\n【美元兑人民币汇率】\n目前美元兑人
民币汇率约为1美元 ≈ 
6.7元人民币。另据摩根资产管理发布的长期资本市场假设报告，由于预期美国通胀持续高企、美元存在贬值压力，而中国通胀较低
且政策制定者希望推动人民币国际化，预计未来10-15年人民币有望继续升值，美元兑人民币可能由约6.7升至5.3左右。\n\n希望以
上信息对您有帮助！'
                        },
                        'id': 'call_-7079982179882628672',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 4603,
                    'output_tokens': 978,
                    'total_tokens': 5581,
                    'input_token_details': {'cache_read': 2624},
                    'output_token_details': {'reasoning': 607}
                }
            ),
            ToolMessage(
                content='数据提取成功',
                name='TextReply',
                id='b36254c8-cc56-4f6b-aaa4-b9e22215145b',
                tool_call_id='call_-7079982179882628672'
            )
        ],
        'structured_response': TextReply(
            reason='用户的请求包含多个任务（天气、新闻、黄金价格、汇率换算），并非单一的天气查询，因此不返回 
WeatherInfo 结构化数据。',
            text='好的，我来逐一回答您的所有问题：\n\n【今日天气】\n杭州今天天气晴朗，气温15摄氏度，非常适合外出活
动。小明可以放心出门和小朋友们一起玩。\n\n【今日新闻】\n根据最新搜索到的新闻：\n1. 
美国方面：美国国会正在审议一项审批法案，该法案将加快对输电线路、天然气管道等多种能源项目的联邦环境审查，同时要求数
据中心为消耗的电力支付数十亿美元的费用，以便让开发商更容易获得运行数据中心所需的电力。\n\n【今日黄金价格】\n国际金
价目前徘徊在14个月低位附近：\n- 现货黄金约每盎司3,845美元（下跌0.3%）\n- 换算下来约为每克123.6美元\n- 
按汇率折算约合人民币每克约828元（国际现货价）\n- 
国内品牌金饰零售价通常会在此基础上加上加工费和品牌溢价，实际购买价格会更高一些\n金价走低的主要原因是美国通胀担忧挥
之不去、市场对美联储进一步收紧货币政策的预期升温，加之科技股下跌带来的压力。\n\n【美元兑人民币汇率】\n目前美元兑人
民币汇率约为1美元 ≈ 
6.7元人民币。另据摩根资产管理发布的长期资本市场假设报告，由于预期美国通胀持续高企、美元存在贬值压力，而中国通胀较低
且政策制定者希望推动人民币国际化，预计未来10-15年人民币有望继续升值，美元兑人民币可能由约6.7升至5.3左右。\n\n希望以
上信息对您有帮助！'
        )
    }
}

--------------------------------------------------


老师案例写法:

In [34]:
class SmartAssistant:
    """多功能智能助手"""
    def __init__(self):
        # 初始化模型
        self.model = model
        # 工具列表
        self.tools = [
            get_weather,
            search_customer_database, 
            tavily_search
        ]
        # 系统提示词
        self.system_prompt = SystemMessage(
            """ 
                你是一个多功能智能助手。请仔细分析用户的每一个请求。
                # 结构化输出规则
                1. **只有**在用户的请求是**单一**的“查询某个城市天气”时，你才能返回 `WeatherInfo` 结构。
                2. 如果用户的请求包含**多个**任务（例如，同时问天气和新闻），或者**不是**查询天气的任务，你必须返回 `TextReply` 结构。
                3. 在 `TextReply` 的 `reason` 字段中，简要说明你为何不返回天气信息（例如："用户请求包含多个任务"）。
                4. 在 `TextReply` 的 `text` 字段中，用自然语言逐一、完整地回答用户的所有问题。
            """
        )
        # 创建智能体
        self.agent = create_agent(
            model=self.model,
            tools=self.tools,
            system_prompt=self.system_prompt,
            response_format=ToolStrategy(
                Union[WeatherInfo, TextReply],
                handle_errors=True,
                # 将最后一条 ToolMessage 的 content 改为简单文字输出, 默认情况下返回 Returning structured response: + 响应的所有内容
                tool_message_content="数据提取成功"
            )
        )
        # 对话历史
        self.messages = []

    def chat(self, user_input: str) -> str:
        """对话接口"""
        # 添加用户消息
        self.messages.append({"role": "user", "content": user_input})
        # 调用 agent
        result = self.agent.invoke({"messages": self.messages})
        # 更新消息历史
        self.messages = result["messages"]
        # 返回最后一条 AI 消息
        for msg in reversed(self.messages):
            rprint(msg, msg.content)
            if msg.type == "ai":
                # 1. 检查有没有直接的内容（比如闲聊）
                if msg.content:
                    return msg.content
                
                # 2. 检查有没有调用工具（比如 TextReply）
                if msg.tool_calls:
                    # 遍历工具调用，找到 TextReply
                    for tool_call in msg.tool_calls:
                        if tool_call['name'] == 'TextReply':
                            # 从 args 里提取出 text 字段
                            args = tool_call['args']
                            # 这里的 text 就是 AI 提取出来的黄金价格回答
                            if 'text' in args:
                                return args['text']
                                # 或者你可以提取 reason: args['reason']
        return "抱歉, 我无法处理这个请求."

    def reset(self):
        """重置对话历史"""
        self.messages = []

主程序

In [35]:
def main():
    assistant = SmartAssistant()

    response = assistant.chat("黄金今天多少钱一克")

    rprint(response)

main()

ToolMessage(
    content='数据提取成功',
    name='TextReply',
    id='9f1394c6-cf75-4e6d-9342-c4efdae08407',
    tool_call_id='call_-7079986921526519330'
)
数据提取成功

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 618,
            'prompt_tokens': 3142,
            'total_tokens': 3760,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 303,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 2880}
        },
        'model_provider': 'openai',
        'model_name': 'glm-5.3',
        'system_fingerprint': None,
        'id': '2026111715011181aecb23188440a2',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a1e84f-43c0-76c3-ae95-212349089b7d-0',
    tool_calls=[
        {
            'name': 'TextReply',
            'args': {
                'reason': 
'用户询问的是黄金价格，并非查询某个城市的天气，因此不返回天气信息结构，而以自然语言回复。',
                'text': '关于今日黄金价格，为您整理如下（金价实时波动，以下仅供参考）：\n\n**国际现货黄金**\n- 
目前国际金价约为每盎司 3,845～3,887 美元，处于近14个月低位附近；\n- 换算成克价：1盎司 ≈ 31.1克，约合 **123～125 
美元/克**；\n- 按人民币汇率（约7.1）折算，约合 **875～888 元/克**（国际现货基准价）。\n\n**国内价格参考**\n- 
上海黄金交易所 Au9999 基础金价通常与国际现货价接近；\n- 
品牌金店（如周大福、老凤祥等）的足金首饰零售价一般会在基础金价上加收工费和品牌溢价，通常比基础金价高出几十到上百元/
克，实际购买时以门店当日挂牌价为准。\n\n**近期走势提示**：受美国通胀担忧、美联储加息预期升温及美元走强等因素影响，
金价近期承压回落，目前徘徊在14个月低位附近。\n\n如需准确购买，建议您查询上海黄金交易所官网、各大金店官网或当日实时
行情获取最新报价。请问还有其他可以帮您的吗？'
            },
            'id': 'call_-7079986921526519330',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 3142,
        'output_tokens': 618,
        'total_tokens': 3760,
        'input_token_details': {'cache_read': 2880},
        'output_token_details': {'reasoning': 303}
    }
)

关于今日黄金价格，为您整理如下（金价实时波动，以下仅供参考）：

**国际现货黄金**
- 目前国际金价约为每盎司 3,845～3,887 美元，处于近14个月低位附近；
- 换算成克价：1盎司 ≈ 31.1克，约合 **123～125 美元/克**；
- 按人民币汇率（约7.1）折算，约合 **875～888 元/克**（国际现货基准价）。

**国内价格参考**
- 上海黄金交易所 Au9999 基础金价通常与国际现货价接近；
- 
品牌金店（如周大福、老凤祥等）的足金首饰零售价一般会在基础金价上加收工费和品牌溢价，通常比基础金价高出几十到上百元/
克，实际购买时以门店当日挂牌价为准。

**近期走势提示**：受美国通胀担忧、美联储加息预期升温及美元走强等因素影响，金价近期承压回落，目前徘徊在14个月低位附
近。

如需准确购买，建议您查询上海黄金交易所官网、各大金店官网或当日实时行情获取最新报价。请问还有其他可以帮您的吗？